# Clustering Comparison: Breast Cancer Wisconsin (Diagnostic)

Dataset: UCI Breast Cancer Wisconsin, loaded with `sc4020.datasets.load_breast_cancer()` (569
samples, 30 continuous features from digitized FNA images of breast masses, standardized).
Ground truth is the binary diagnosis: `0` Benign (357), `1` Malignant (212).

The same seven algorithms as the Wine model-zoo notebook, with `k=2`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import load_breast_cancer
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "ari",
]

## Load and standardize

The tracking columns `id` and `Unnamed: 32` are dropped, `diagnosis` becomes the ground truth
(`B→0`, `M→1`) and the 30 remaining features are standardized.

In [ ]:
ds = load_breast_cancer()
print(f"{ds.name}: X={ds.data.shape}, classes={ds.target_names}, n_true_clusters={ds.n_true_clusters}")
print(pd.Series(ds.target).map(dict(enumerate(ds.target_names))).value_counts())

## 2D embedding for visualization

Clustering runs on the full standardized 30-D space; PCA to 2D is *only* used to plot.

In [ ]:
X_2d, xlabel, ylabel = gf.project_2d(ds.data)
print(xlabel, ylabel)

## Fit 7 clustering models

* **By the shared rule** (`tuning.select_all`): K-Means and Gaussian Mixture at k=2 (GMM covariance
  type by BIC); DBSCAN and HDBSCAN at the setting with the best silhouette among those with at least
  2 clusters and at most 50% noise.
* **Hierarchical:** Agglomerative (k=2, Ward linkage)
* **Graph-based:** Spectral (k=2, nearest-neighbours affinity)
* **Exemplar-based:** Affinity Propagation (damping=0.9, preference=-150)

In [ ]:
# K-Means, DBSCAN, HDBSCAN and GMM by the shared rule (tuning.select_all), with K-Means and GMM at
# the number of classes; Agglomerative (Ward) and Spectral at the same k; Affinity Propagation
# picks its own number of exemplars.
k = ds.n_true_clusters
selections = tuning.select_all(ds.data, k=k, y_true=ds.target)
params = {m: s.params for m, s in selections.items()}
params["Agglomerative"] = {"n_clusters": k, "linkage": "ward"}
params["Spectral"] = {"n_clusters": k, "affinity": "nearest_neighbors"}
params["Affinity Propagation"] = {"damping": 0.9, "preference": -150}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:22s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics

In [ ]:
results_df = gf.build_results_table("breast_cancer", ds.data, results, ds.target)
display(results_df[METRIC_COLS].round(4))
results_df.to_csv(results_dir("datasets") / "breast_cancer_results.csv", index=False)

## Cluster visualizations vs. ground truth

In [ ]:
gf.plot_cluster_grid(
    ds.data, results, y_true=ds.target, class_names=ds.target_names, scores=gf.score_labels(results_df),
    title="Ground truth and 7 clustering methods (Breast Cancer; 0 benign, 1 malignant)", ncols=4,
)
plt.show()

## Discussion

* **GMM (ARI ≈ 0.77) and Spectral (≈ 0.76) are best:** benign and malignant cells form two
  distributions with different spreads, which GMM's per-cluster covariance and the
  nearest-neighbors graph capture better than K-Means (≈ 0.65).
* **Agglomerative (Ward, ≈ 0.58):** minimizing within-cluster variance mirrors K-Means but
  is less accurate here.
* **Curse of dimensionality in DBSCAN:** in 30-D, pairwise Euclidean distances become sparse.
  A single global ε either collapses everything into one cluster or discards the dispersed
  malignant points as noise (`-1`).
* **Affinity Propagation:** without a fixed cluster count, message passing finds many local
  sub-densities and over-fragments the two diagnostic classes.